In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [3]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

df = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train: {df.shape}  |  Test: {test.shape}')
print(f'Missing values in train:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'Missing values in test:\n{test.isnull().sum()[test.isnull().sum() > 0]}')

Train: (699635, 23)  |  Test: (299844, 22)
Missing values in train:
Arrival Delay in Minutes    292
dtype: int64
Missing values in test:
Arrival Delay in Minutes    130
dtype: int64


In [4]:
X = df.drop(columns=["id", "satisfaction"], errors="ignore")
y = df["satisfaction"]

numerical_cols = [
    "Age", "Flight Distance",
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location", "Food and drink",
    "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service",
    "Leg room service", "Baggage handling",
    "Checkin service", "Cleanliness",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

categorical_cols = [
    "Gender", "Type of Travel", "Class", "Customer Type"
]

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [5]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [6]:
catboost_l2_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=10,       # L2 regularization
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        allow_writing_files=False
    ))
])

catboost_l2_model.fit(X_train, y_train)

print("CatBoost with L2 regularization trained successfully!")

0:	total: 199ms	remaining: 4m 58s
100:	total: 4.49s	remaining: 1m 2s
200:	total: 8.83s	remaining: 57.1s
300:	total: 12.8s	remaining: 50.9s
400:	total: 16.9s	remaining: 46.4s
500:	total: 21.2s	remaining: 42.4s
600:	total: 25.4s	remaining: 38s
700:	total: 29.9s	remaining: 34.1s
800:	total: 33.9s	remaining: 29.6s
900:	total: 37.8s	remaining: 25.2s
1000:	total: 42.2s	remaining: 21.1s
1100:	total: 46s	remaining: 16.7s
1200:	total: 50.2s	remaining: 12.5s
1300:	total: 54s	remaining: 8.26s
1400:	total: 57.5s	remaining: 4.07s
1499:	total: 1m 1s	remaining: 0us
CatBoost with L2 regularization trained successfully!


In [7]:
y_pred = catboost_l2_model.predict(X_test).ravel().astype(int)
y_prob = catboost_l2_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(target_encoder.classes_)),
    target_names=[str(c) for c in target_encoder.classes_],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9247107420297727

Classification Report:
              precision    recall  f1-score   support

       False       0.92      0.94      0.93     77859
        True       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927

Confusion Matrix:
[[73364  4495]
 [ 6040 56028]]
ROC-AUC: 0.9574508236083669


In [10]:
new_data = pd.read_csv("test.csv")

ids = new_data["id"].copy()

X_new = new_data.drop(
    columns=["id", "satisfaction"],
    errors="ignore"
)

predictions = catboost_l2_model.predict(X_new).ravel().astype(int)
predicted_labels = target_encoder.inverse_transform(predictions)

probabilities = catboost_l2_model.predict_proba(X_new)

results = pd.DataFrame({
    "id": ids,
    "Predicted_Satisfaction": predicted_labels,
    "Probability_Class_0": probabilities[:, 0],
    "Probability_Class_1": probabilities[:, 1]
})

results[['id','Predicted_Satisfaction']].to_csv("catboost_l2_predictions.csv", index=False)

print(results.head())
print("Saved to catboost_l2_predictions.csv")

       id  Predicted_Satisfaction  Probability_Class_0  Probability_Class_1
0  699635                   False             0.742505             0.257495
1  699636                    True             0.111243             0.888757
2  699637                    True             0.122421             0.877579
3  699638                   False             0.935548             0.064452
4  699639                   False             0.941950             0.058050
Saved to catboost_l2_predictions.csv


In [12]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [13]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train), start=1
):
    print(f"\n========== Fold {fold} ==========")

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train[train_idx]
    y_fold_val = y_train[val_idx]

    # Create a fresh pipeline for every fold
    fold_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", CatBoostClassifier(
            iterations=1500,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=10,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=0,
            allow_writing_files=False,
            thread_count=-1
        ))
    ])

    fold_model.fit(X_fold_train, y_fold_train)

    y_val_pred = fold_model.predict(X_fold_val).ravel().astype(int)
    y_val_prob = fold_model.predict_proba(X_fold_val)[:, 1]

    metrics = {
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, y_val_pred),
        "Precision": precision_score(
            y_fold_val, y_val_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val, y_val_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_fold_val, y_val_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_fold_val, y_val_prob)
    }

    fold_results.append(metrics)

    print(pd.Series(metrics).to_string())


========== Fold 1 ==========
Fold         1.000000
Accuracy     0.927275
Precision    0.928545
Recall       0.905750
F1 Score     0.917006
ROC-AUC      0.958809

========== Fold 2 ==========
Fold         2.000000
Accuracy     0.925461
Precision    0.924075
Recall       0.906433
F1 Score     0.915169
ROC-AUC      0.958124

========== Fold 3 ==========
Fold         3.000000
Accuracy     0.925801
Precision    0.926620
Recall       0.904338
F1 Score     0.915343
ROC-AUC      0.959125

========== Fold 4 ==========
Fold         4.000000
Accuracy     0.926979
Precision    0.926591
Recall       0.907258
F1 Score     0.916823
ROC-AUC      0.958942

========== Fold 5 ==========
Fold         5.000000
Accuracy     0.925434
Precision    0.927494
Recall       0.902445
F1 Score     0.914798
ROC-AUC      0.957836


In [14]:
cv_results = pd.DataFrame(fold_results)

print("\nIndividual Fold Results:")
print(cv_results.round(4))

print("\nMean Cross-Validation Scores:")
print(cv_results.drop(columns="Fold").mean().round(4))

print("\nStandard Deviation:")
print(cv_results.drop(columns="Fold").std().round(4))


Individual Fold Results:
   Fold  Accuracy  Precision  Recall  F1 Score  ROC-AUC
0     1    0.9273     0.9285  0.9057    0.9170   0.9588
1     2    0.9255     0.9241  0.9064    0.9152   0.9581
2     3    0.9258     0.9266  0.9043    0.9153   0.9591
3     4    0.9270     0.9266  0.9073    0.9168   0.9589
4     5    0.9254     0.9275  0.9024    0.9148   0.9578

Mean Cross-Validation Scores:
Accuracy     0.9262
Precision    0.9267
Recall       0.9052
F1 Score     0.9158
ROC-AUC      0.9586
dtype: float64

Standard Deviation:
Accuracy     0.0009
Precision    0.0017
Recall       0.0019
F1 Score     0.0010
ROC-AUC      0.0006
dtype: float64


In [15]:
catboost_l2_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=10,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        allow_writing_files=False,
        thread_count=-1
    ))
])

catboost_l2_model.fit(X_train, y_train)

y_test_pred = (
    catboost_l2_model.predict(X_test).ravel().astype(int)
)
y_test_prob = catboost_l2_model.predict_proba(X_test)[:, 1]

print("\nTest Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_prob))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=np.arange(len(target_encoder.classes_)),
    target_names=[str(c) for c in target_encoder.classes_],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

0:	total: 37.5ms	remaining: 56.3s
100:	total: 4.7s	remaining: 1m 5s
200:	total: 9.08s	remaining: 58.7s
300:	total: 13.1s	remaining: 52.1s
400:	total: 17.1s	remaining: 46.9s
500:	total: 21.5s	remaining: 42.8s
600:	total: 25.7s	remaining: 38.5s
700:	total: 29.5s	remaining: 33.6s
800:	total: 32.9s	remaining: 28.7s
900:	total: 36.4s	remaining: 24.2s
1000:	total: 40s	remaining: 20s
1100:	total: 44s	remaining: 16s
1200:	total: 48.4s	remaining: 12.1s
1300:	total: 52.7s	remaining: 8.06s
1400:	total: 56.8s	remaining: 4.01s
1499:	total: 1m	remaining: 0us

Test Accuracy: 0.9247107420297727
Test ROC-AUC: 0.9574508236083669

Classification Report:
              precision    recall  f1-score   support

       False       0.92      0.94      0.93     77859
        True       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927

Confusion Matrix:
[[73364  449

In [16]:
import pandas as pd
import numpy as np

# 1. Load new test data (without target)
new_data = pd.read_csv("test.csv")

# 2. Preserve IDs
ids = new_data["id"].copy()

# 3. Prepare features
X_new = new_data.drop(
    columns=["id", "satisfaction"],
    errors="ignore"
)

# 4. Predict satisfaction
predictions = (
    catboost_l2_model.predict(X_new)
    .ravel()
    .astype(int)
)

predicted_labels = target_encoder.inverse_transform(predictions)

# 5. Predict class probabilities
probabilities = catboost_l2_model.predict_proba(X_new)

# 6. Create results DataFrame
results = pd.DataFrame({
    "id": ids,
    "Predicted_Satisfaction": predicted_labels,
    "Probability_Class_0": probabilities[:, 0],
    "Probability_Class_1": probabilities[:, 1]
})

# 7. Display predictions
print(results.head(10))

# 8. Save predictions
results[['id','Predicted_Satisfaction']].to_csv("catboost_l2_kfold_predictions.csv", index=False)

print("Predictions saved successfully!")

       id  Predicted_Satisfaction  Probability_Class_0  Probability_Class_1
0  699635                   False             0.742505             0.257495
1  699636                    True             0.111243             0.888757
2  699637                    True             0.122421             0.877579
3  699638                   False             0.935548             0.064452
4  699639                   False             0.941950             0.058050
5  699640                    True             0.039900             0.960100
6  699641                   False             0.972430             0.027570
7  699642                    True             0.078266             0.921734
8  699643                    True             0.016623             0.983377
9  699644                   False             0.977773             0.022227
Predictions saved successfully!


In [17]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import optuna
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 5-fold cross-validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# Create a fresh preprocessor for each fold
def build_preprocessor():
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])


# Optuna objective function
def objective(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations", 500, 1500, step=250
        ),
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1.0, 30.0, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 2.0
        ),
        "border_count": trial.suggest_categorical(
            "border_count", [64, 128, 254]
        )
    }

    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train), start=1
    ):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = np.asarray(y_train)[train_idx]
        y_fold_val = np.asarray(y_train)[val_idx]

        # Fit preprocessing only on this fold's training data
        preprocessor = build_preprocessor()

        X_fold_train_processed = preprocessor.fit_transform(
            X_fold_train
        )
        X_fold_val_processed = preprocessor.transform(
            X_fold_val
        )

        model = CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        )

        model.fit(
            X_fold_train_processed,
            y_fold_train,
            eval_set=(
                X_fold_val_processed,
                y_fold_val
            ),
            early_stopping_rounds=100,
            verbose=False
        )

        val_probabilities = model.predict_proba(
            X_fold_val_processed
        )[:, 1]

        score = roc_auc_score(
            y_fold_val,
            val_probabilities
        )

        fold_scores.append(score)

    # Optuna maximizes mean ROC-AUC
    return float(np.mean(fold_scores))


# Create and run the study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=20
)

print("\nBest Mean CV ROC-AUC:", study.best_value)
print("\nBest Hyperparameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

[I 2026-10-16 12:58:19,541] A new study created in memory with name: no-name-6538391c-e5c5-40c6-a145-16802b5f7ab1
[I 2026-10-16 13:01:00,209] Trial 0 finished with value: 0.9586975628795674 and parameters: {'iterations': 750, 'depth': 8, 'learning_rate': 0.07259248719561363, 'l2_leaf_reg': 7.6611007077713635, 'random_strength': 0.31203728088487304, 'border_count': 254}. Best is trial 0 with value: 0.9586975628795674.
[I 2026-10-16 13:06:30,494] Trial 1 finished with value: 0.9574646160768528 and parameters: {'iterations': 1250, 'depth': 7, 'learning_rate': 0.010573268083515799, 'l2_leaf_reg': 27.08160864249967, 'random_strength': 1.6648852816008435, 'border_count': 64}. Best is trial 0 with value: 0.9586975628795674.
[I 2026-10-16 13:10:11,978] Trial 2 finished with value: 0.9578957768303988 and parameters: {'iterations': 750, 'depth': 6, 'learning_rate': 0.032211189352044464, 'l2_leaf_reg': 2.6926552514864723, 'random_strength': 1.223705789444759, 'border_count': 254}. Best is trial 0

In [ ]:
trials_df = study.trials_dataframe()

print(
    trials_df[
        ["number", "value", "params_depth",
         "params_learning_rate", "params_l2_leaf_reg"]
    ]
    .sort_values("value", ascending=False)
    .head(10)
)

In [ ]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()

vis.plot_param_importances(study).show()

In [ ]:
best_params = study.best_params.copy()

catboost_l2_model = Pipeline([
    ("preprocessor", build_preprocessor()),

    ("classifier", CatBoostClassifier(
        **best_params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        allow_writing_files=False,
        thread_count=-1
    ))
])

catboost_l2_model.fit(X_train, y_train)

print("Final tuned CatBoost model trained!")

In [ ]:
y_test_pred = (
    catboost_l2_model.predict(X_test)
    .ravel()
    .astype(int)
)

y_test_prob = catboost_l2_model.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test F1:", f1_score(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_prob))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))